# 🐕 PPO Unitree Go2 — v4.1

| Celda | Qué hace | Cuándo correrla |
|---|---|---|
| 1 | Dependencias | Siempre, primera |
| 2 | Modelo Go2 | Siempre |
| 3 | Entorno | Siempre |
| 4 | Drive + rutas | Siempre |
| 5 | Entrenar (detecta solo si reanudar o empezar) | Cuando se quiere entrenar |
| 6 | Descargar política para evaluar en PyCharm | Cuando se quiere renderizar el video en pc local |

---
## CELDA 1 — Dependencias
⚠️ Siempre primera.

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'

import subprocess
subprocess.run(['pip', 'install', '-q', 'mujoco', 'stable-baselines3[extra]',
                'gymnasium', 'imageio[ffmpeg]'], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], capture_output=True)
print('✅ Dependencias instaladas')

---
## CELDA 2 — Modelo Go2
⚠️ Siempre (filesystem efímero).

In [ ]:
import subprocess, os

if not os.path.exists('mujoco_menagerie'):
    subprocess.run([
        'git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
        'https://github.com/google-deepmind/mujoco_menagerie.git'
    ], check=True)
    subprocess.run([
        'git', '-C', 'mujoco_menagerie', 'sparse-checkout', 'set', 'unitree_go2'
    ], check=True)

MODEL_PATH = 'mujoco_menagerie/unitree_go2/scene.xml'
print(f'✅ Modelo listo: {MODEL_PATH}')

---
## CELDA 3 — Entorno Go2
⚠️ Siempre.

In [ ]:
import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces


class Go2LocomotionEnv(gym.Env):
    metadata = {'render_modes': ['rgb_array'], 'render_fps': 50}

    def __init__(self, render_mode=None, max_steps=1000):
        super().__init__()
        self.render_mode = render_mode
        self.max_steps   = max_steps
        self.model       = mujoco.MjModel.from_xml_path(MODEL_PATH)
        self.data        = mujoco.MjData(self.model)
        self.ctrl_range  = self.model.actuator_ctrlrange
        self.init_qpos   = self.model.keyframe('home').qpos.copy() if self.model.nkey > 0 else None

        obs_dim = self._get_obs().shape[0]
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Box(
            self.ctrl_range[:, 0].astype(np.float32),
            self.ctrl_range[:, 1].astype(np.float32),
            dtype=np.float32
        )
        self._step_count = 0

    def _get_obs(self):
        return np.concatenate([
            self.data.qpos[7:], self.data.qvel, self.data.qpos[3:7]
        ]).astype(np.float32)

    def _compute_reward(self):
        vx             = self.data.qvel[0]
        energy_penalty = 0.001 * np.sum(np.square(self.data.ctrl))
        height         = self.data.qpos[2]
        height_penalty = 2.0 * max(0, 0.25 - height)
        w, x, y, z    = self.data.qpos[3:7]
        up_z           = 1 - 2 * (x*x + y*y)
        orient_penalty = 0.5 * max(0, 0.5 - up_z)
        return vx - energy_penalty - height_penalty - orient_penalty

    def _is_terminated(self):
        return bool(self.data.qpos[2] < 0.18)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        if self.init_qpos is not None:
            self.data.qpos[:] = self.init_qpos
            self.data.qpos[7:] += self.np_random.uniform(-0.05, 0.05, size=12)
        mujoco.mj_forward(self.model, self.data)
        self._step_count = 0
        return self._get_obs(), {}

    def step(self, action):
        self.data.ctrl[:] = np.clip(action, self.ctrl_range[:, 0], self.ctrl_range[:, 1])
        for _ in range(4):
            mujoco.mj_step(self.model, self.data)
        self._step_count += 1
        obs        = self._get_obs()
        reward     = self._compute_reward()
        terminated = self._is_terminated()
        truncated  = self._step_count >= self.max_steps
        return obs, reward, terminated, truncated, {}

    def close(self):
        pass


env = Go2LocomotionEnv()
obs, _ = env.reset()
print(f'✅ Entorno OK — obs: {obs.shape} | acciones: {env.action_space.shape}')
env.close()

---
## CELDA 4 — Drive y rutas
⚠️ Siempre.

In [ ]:
from google.colab import drive
import os, glob

drive.mount('/content/drive')

DRIVE_DIR      = '/content/drive/MyDrive/go2_ppo'
CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
TB_DIR         = f'{DRIVE_DIR}/tb_logs'

for d in [DRIVE_DIR, CHECKPOINT_DIR, TB_DIR]:
    os.makedirs(d, exist_ok=True)


def find_latest_checkpoint():
    """Retorna (policy_path, vecnorm_path, steps) del checkpoint más reciente, o None si no hay."""
    # Buscar en checkpoints local y en Drive
    zips = glob.glob(f'/content/checkpoints/go2_ppo_*.zip') + \
           glob.glob(f'{CHECKPOINT_DIR}/go2_ppo_*.zip')
    if not zips:
        return None
    # Ordenar por pasos
    zips = list(set(zips))  # deduplicar
    zips.sort(key=lambda p: int(p.split('go2_ppo_')[1].split('_steps')[0]))
    latest_zip = zips[-1]
    steps = int(latest_zip.split('go2_ppo_')[1].split('_steps')[0])

    # Buscar vecnormalize: primero el correspondiente, luego cualquiera
    vecnorm_candidates = (
        glob.glob(f'/content/checkpoints/go2_vecnormalize_*.pkl') +
        glob.glob(f'{CHECKPOINT_DIR}/go2_vecnormalize_*.pkl') +
        glob.glob(f'{DRIVE_DIR}/go2_vecnormalize*.pkl')
    )
    vecnorm_candidates = list(set(vecnorm_candidates))
    vecnorm_candidates.sort()
    vecnorm_path = vecnorm_candidates[-1] if vecnorm_candidates else None

    return latest_zip, vecnorm_path, steps


result = find_latest_checkpoint()
if result:
    _, _, steps = result
    print(f'✅ Drive montado — checkpoint más reciente: {steps:,} pasos')
else:
    print('✅ Drive montado — sin checkpoints previos (entrenamiento nuevo)')

---
## CELDA 5 — Entrenar

**Una sola celda.** Detecta automáticamente si hay checkpoint previo y retoma desde ahí.

- `TOTAL_TIMESTEPS` es el **objetivo final total**, no pasos adicionales.
- Si ya hay 2M pasos y se cambia `TOTAL_TIMESTEPS = 5_000_000`, entrena 3M más.
- Si no hay nada, empieza desde cero.
- `SAVE_FREQ`: cada cuántos pasos guarda checkpoint en Drive.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback
import torch, os, shutil, glob

# ── Configuración ──────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 5_000_000   # objetivo final total
SAVE_FREQ       = 200_000     # guardar en Drive cada N pasos
N_ENVS          = 8
# ───────────────────────────────────────────────────────────────────────────

LOCAL_CKPT = '/content/checkpoints'
os.makedirs(LOCAL_CKPT, exist_ok=True)


class SaveCallback(BaseCallback):
    """Guarda checkpoint local cada N pasos. Drive se actualiza al terminar."""
    def __init__(self, save_freq_steps, save_dir, vec_env_ref):
        super().__init__(verbose=0)
        self.save_freq_steps = save_freq_steps
        self.save_dir        = save_dir
        self.vec_env_ref     = vec_env_ref
        self._last_saved     = 0

    def _on_step(self):
        # Usar total_timesteps acumulados para decidir cuándo guardar
        if self.num_timesteps - self._last_saved >= self.save_freq_steps:
            self._last_saved = self.num_timesteps
            path = f'{self.save_dir}/go2_ppo_{self.num_timesteps}_steps'
            self.model.save(path)
            self.vec_env_ref.save(f'{self.save_dir}/go2_vecnormalize_{self.num_timesteps}_steps.pkl')
            print(f'\n💾 Checkpoint local: {self.num_timesteps:,} pasos')
        return True


# ── Detectar checkpoint previo ─────────────────────────────────────────────
checkpoint_info = find_latest_checkpoint()

if checkpoint_info is None:
    # ── Entrenamiento desde cero ───────────────────────────────────────────
    print(f'🆕 Sin checkpoint previo — entrenando desde cero')
    print(f'   Objetivo: {TOTAL_TIMESTEPS:,} pasos\n')

    vec_env = make_vec_env(Go2LocomotionEnv, n_envs=N_ENVS)
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    model = PPO(
        policy='MlpPolicy',
        env=vec_env,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        learning_rate=3e-4,
        policy_kwargs=dict(
            net_arch=dict(pi=[256, 256], vf=[256, 256]),
            activation_fn=torch.nn.ELU,
        ),
        verbose=1,
        tensorboard_log=TB_DIR,
        device='cpu',
    )
    steps_done = 0

else:
    # ── Reanudar desde checkpoint ──────────────────────────────────────────
    policy_path, vecnorm_path, steps_done = checkpoint_info
    remaining = TOTAL_TIMESTEPS - steps_done

    if remaining <= 0:
        print(f'✅ Ya se alcanzaron {steps_done:,} pasos.')
        print(f'   Aumentá TOTAL_TIMESTEPS (actualmente {TOTAL_TIMESTEPS:,}) para continuar.')
        raise SystemExit

    print(f'🔄 Checkpoint encontrado: {steps_done:,} pasos')
    print(f'   Objetivo: {TOTAL_TIMESTEPS:,} | Restantes: {remaining:,}\n')

    vec_env = make_vec_env(Go2LocomotionEnv, n_envs=N_ENVS)
    vec_env = VecNormalize.load(vecnorm_path, vec_env)
    vec_env.training    = True
    vec_env.norm_reward = True

    model = PPO.load(
        policy_path,
        env=vec_env,
        device='cpu',
        tensorboard_log=TB_DIR,
    )


# ── Entrenar ───────────────────────────────────────────────────────────────
remaining = TOTAL_TIMESTEPS - steps_done
callback  = SaveCallback(SAVE_FREQ, LOCAL_CKPT, vec_env)

model.learn(
    total_timesteps=remaining,
    callback=callback,
    reset_num_timesteps=(steps_done == 0),  # False si estamos reanudando
    progress_bar=False,
)

# ── Guardar en Drive al terminar ───────────────────────────────────────────
print('\n📤 Copiando checkpoints a Drive...')
for f in glob.glob(f'{LOCAL_CKPT}/*'):
    dest = f'{CHECKPOINT_DIR}/{os.path.basename(f)}'
    shutil.copy(f, dest)
print(f'✅ Entrenamiento completo — {TOTAL_TIMESTEPS:,} pasos totales')

---
## CELDA 6 — Preparar archivos para evaluar en PyCharm

Descargar el checkpoint más reciente y el vecnormalize a la PC.
Después correr `evaluate.py` en PyCharm para generar el video.

In [ ]:
from google.colab import files
import glob, os

result = find_latest_checkpoint()
if result is None:
    print('No hay checkpoints todavía.')
else:
    policy_path, vecnorm_path, steps = result
    print(f'📥 Descargando checkpoint de {steps:,} pasos...')
    files.download(policy_path)
    files.download(vecnorm_path)
    print('✅ Listo — copiar ambos archivos a la carpeta del proyecto en PyCharm y correr evaluate.py')